In [1]:
!pip install --upgrade --quiet langchain langchain-community langchain-experimental langchain-groq langchain-neo4j neo4j


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


1. Connect to Neo4j

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("URI:", os.getenv("NEO4J_URI"))

URI: neo4j+s://579b613f.databases.neo4j.io


In [3]:
from langchain_neo4j import Neo4jGraph


NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
GROQ_API_KEY = os.environ["GROQ_API_KEY"]



In [4]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print("Connected to Neo4j")

Connected to Neo4j


2. Create the LLM

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

3. Create a tiny movie dataset


In [6]:
from langchain_core.documents import Document

movie_texts = [
    "J.R.R. Tolkien wrote The Hobbit. The Hobbit is a Fantasy book published in 1937. Peter Jackson directed the film adaptation of The Hobbit. Martin Freeman acted in The Hobbit.",
    "J.R.R. Tolkien wrote The Lord of the Rings. The Lord of the Rings is a Fantasy book published in 1954. Peter Jackson directed the film adaptation of The Lord of the Rings. Elijah Wood acted in The Lord of the Rings.",
    "Peter Jackson directed King Kong. King Kong is an Adventure film released in 2005. Andy Serkis acted in King Kong. Andy Serkis also acted in The Lord of the Rings."
]

documents = [Document(page_content=text) for text in movie_texts]

documents

[Document(metadata={}, page_content='J.R.R. Tolkien wrote The Hobbit. The Hobbit is a Fantasy book published in 1937. Peter Jackson directed the film adaptation of The Hobbit. Martin Freeman acted in The Hobbit.'),
 Document(metadata={}, page_content='J.R.R. Tolkien wrote The Lord of the Rings. The Lord of the Rings is a Fantasy book published in 1954. Peter Jackson directed the film adaptation of The Lord of the Rings. Elijah Wood acted in The Lord of the Rings.'),
 Document(metadata={}, page_content='Peter Jackson directed King Kong. King Kong is an Adventure film released in 2005. Andy Serkis acted in King Kong. Andy Serkis also acted in The Lord of the Rings.')]

4. Chunk the documents

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)


Chunk 1: J.R.R. Tolkien wrote The Hobbit. The Hobbit is a Fantasy book published in 1937. Peter Jackson directed the film adaptation of The Hobbit. Martin Freeman acted in The Hobbit.
Chunk 2: J.R.R. Tolkien wrote The Lord of the Rings. The Lord of the Rings is a Fantasy book published in 1954. Peter Jackson directed the film adaptation of The Lord of the Rings. Elijah Wood acted in The
Chunk 3: Wood acted in The Lord of the Rings.
Chunk 4: Peter Jackson directed King Kong. King Kong is an Adventure film released in 2005. Andy Serkis acted in King Kong. Andy Serkis also acted in The Lord of the Rings.


5. Convert chunks into graph data

In [8]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)


/var/folders/wl/t32664pd5433qm9mdtfwtdxc0000gn/T/ipykernel_73495/3834549908.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [9]:
graph_documents

[GraphDocument(nodes=[Node(id='J.R.R. Tolkien', type='Person', properties={}), Node(id='Peter Jackson', type='Person', properties={}), Node(id='Martin Freeman', type='Person', properties={}), Node(id='The Hobbit', type='Movie', properties={}), Node(id='Fantasy', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Peter Jackson', type='Person', properties={}), target=Node(id='The Hobbit', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Martin Freeman', type='Person', properties={}), target=Node(id='The Hobbit', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='The Hobbit', type='Movie', properties={}), target=Node(id='Fantasy', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='J.R.R. Tolkien wrote The Hobbit. The Hobbit is a Fantasy book published in 1937. Peter Jackson directed the film adaptation of The Hobbit. Martin Fre

In [10]:
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)



--- Graph Document 1 ---
Nodes:
id='J.R.R. Tolkien' type='Person' properties={}
id='Peter Jackson' type='Person' properties={}
id='Martin Freeman' type='Person' properties={}
id='The Hobbit' type='Movie' properties={}
id='Fantasy' type='Genre' properties={}
Relationships:
source=Node(id='Peter Jackson', type='Person', properties={}) target=Node(id='The Hobbit', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Martin Freeman', type='Person', properties={}) target=Node(id='The Hobbit', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='The Hobbit', type='Movie', properties={}) target=Node(id='Fantasy', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='J.R.R. Tolkien' type='Person' properties={}
id='Peter Jackson' type='Person' properties={}
id='Elijah Wood' type='Person' properties={}
id='The Lord Of The Rings' type='Movie' properties={}
id='Fantasy' type='Genre' properties={}
Relationships:


6. Store the graph in Neo4j


In [11]:

graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

print("Graph data added to Neo4j")

Graph data added to Neo4j


In [12]:
graph.query("""
MATCH (p:Person) 
WHERE p.name IS NULL 
SET p.name = p.id
""")

graph.query("""
MATCH (m:Movie) 
WHERE m.title IS NULL 
SET m.title = m.id
""")

print("Nodes aligned successfully!")

Nodes aligned successfully!


7. Refresh and inspect the Neo4j schema


In [13]:

graph.refresh_schema()
print(graph.schema)

Node properties:
Person {name: STRING, id: STRING}
Movie {title: STRING, id: STRING}
Document {id: STRING, text: STRING}
Genre {id: STRING}
Book {title: STRING, id: STRING}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:ACTED_IN]->(:Book)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:WROTE]->(:Book)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:DIRECTED]->(:Book)
(:Movie)-[:IN_GENRE]->(:Genre)
(:Movie)-[:ADAPTED_INTO]->(:Movie)
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Genre)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Book)
(:Book)-[:IN_GENRE]->(:Genre)
(:Book)-[:ADAPTED_INTO]->(:Movie)


8. Retrieval: ask Neo4j for connected facts

In [14]:

result = graph.query("""
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
OPTIONAL MATCH (actor:Person)-[:ACTED_IN]->(movie)
RETURN director.id AS director, movie.id AS movie, collect(actor.id) AS actors
""")

result

[{'director': 'Peter Jackson',
  'movie': 'The Hobbit',
  'actors': ['Martin Freeman']},
 {'director': 'Peter Jackson',
  'movie': 'The Hobbit (Film)',
  'actors': ['Martin Freeman']},
 {'director': 'Peter Jackson',
  'movie': 'The Lord Of The Rings',
  'actors': ['Wood', 'Elijah Wood', 'Andy Serkis']},
 {'director': 'Peter Jackson',
  'movie': 'The Lord Of The Rings (Film)',
  'actors': ['Elijah Wood']},
 {'director': 'Peter Jackson',
  'movie': 'King Kong',
  'actors': ['Andy Serkis']}]

9. Generation: let the LLM answer using the graph

In [15]:
from langchain_core.prompts import PromptTemplate

QA_TEMPLATE = """Based on the provided context, answer the question directly. 
If the context lists movies, include them in your answer.

Context:
{context}

Question:
{question}
"""

QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"], 
    template=QA_TEMPLATE
)

In [16]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    qa_prompt=QA_PROMPT,
    verbose=True,
    allow_dangerous_requests=True
)


In [17]:
response = chain.invoke({
    "query": "Which movies did Peter Jackson direct?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "Peter Jackson"})-[:DIRECTED]->(m:Movie)
RETURN m.title, m.id
Full Context:
[{'m.title': 'The Hobbit', 'm.id': 'The Hobbit'}, {'m.title': 'The Hobbit (Film)', 'm.id': 'The Hobbit (Film)'}, {'m.title': 'The Lord Of The Rings', 'm.id': 'The Lord Of The Rings'}, {'m.title': 'The Lord Of The Rings (Film)', 'm.id': 'The Lord Of The Rings (Film)'}, {'m.title': 'King Kong', 'm.id': 'King Kong'}]

> Finished chain.
Peter Jackson directed the following movies in the list:

- **The Hobbit (Film)**
- **The Lord Of The Rings (Film)**
- **King Kong**


In [18]:
questions = [
    "Which films did Peter Jackson direct?",
    "Which actor appeared in both King Kong and The Lord of the Rings?",
    "Which books did J.R.R. Tolkien write?",
    "Who directed the film adaptations of books written by Tolkien?"
]

for q in questions:
    response = chain.invoke({"query": q})
    print(f"Q: {q}")
    print(f"A: {response['result']}\n")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "Peter Jackson"})-[:DIRECTED]->(m:Movie)
RETURN m.title, m.id
Full Context:
[{'m.title': 'The Hobbit', 'm.id': 'The Hobbit'}, {'m.title': 'The Hobbit (Film)', 'm.id': 'The Hobbit (Film)'}, {'m.title': 'The Lord Of The Rings', 'm.id': 'The Lord Of The Rings'}, {'m.title': 'The Lord Of The Rings (Film)', 'm.id': 'The Lord Of The Rings (Film)'}, {'m.title': 'King Kong', 'm.id': 'King Kong'}]

> Finished chain.
Q: Which films did Peter Jackson direct?
A: Peter Jackson directed the following films listed in the context:

- **The Hobbit (Film)**
- **The Lord Of The Rings (Film)**
- **King Kong**



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m1:Movie {title: 'King Kong'}), (p)-[:ACTED_IN]->(m2:Movie {title: 'The Lord of the Rings'})
RETURN p.name, p.id
Full Context:
[]

> Finished chain.
Q: Which actor appeared in both King Kong and The Lord of the Rings?
A: T